# [A100 40GB] 전체 최종 평가: baseline (Qwen3-VL-4B-Instruct, 학습 없음) — MMMU test 10,500 / MMMU-Pro 1,730 × 2

GRPO v4 step 170 전체 평가와 **같은 문항·같은 생성 설정·같은 코드·같은 GPU 종류**로 base 모델만 바꿔 돌립니다. 두 결과가 있으면 같은 문항에서 개선폭(+/−, McNemar p)을 계산합니다.

**두 계정에 나눠 돌립니다.** 첫 코드 셀의 `PART`만 정하세요.

| `PART` | 계정 | 세트 | 생성 수 | 예상 |
|---|---|---|---|---|
| `'mmmu_test'` | 공용 계정 | MMMU test 전체 | 10,500 | 약 11~14시간 (baseline은 GRPO보다 출력이 길고 잘림이 많음) |
| `'pro'` | 팀원 계정 | MMMU-Pro standard(10지선다) 전체 → vision 전체 | 1,730 × 2 | 약 4~8시간 |

- 모델: `Qwen/Qwen3-VL-4B-Instruct` @ `ebb281ec…`(val baseline 63.89%와 같은 revision). vLLM이 Hugging Face에서 직접 받습니다(병합 없음).
- 문항: `configs/final_eval_ids_full.json`(GRPO 평가와 같은 파일, 해시 확인). 생성 설정 `final_16k.yaml`, seed 3407.
- **같은 계정에서 GRPO 평가가 아직 돌고 있으면 그게 끝난 뒤에 실행하세요.** 결과 폴더가 달라 섞이지는 않지만, 동시에 돌리면 유닛이 두 배로 빠집니다.
- **끊겨도 됩니다:** 묶음마다 Drive에 저장합니다. 런타임이 끊기면 **같은 PART로 처음부터 다시 실행**하세요. 저장된 문항은 건너뛰고 preflight도 재사용합니다(재개는 40GB에서만).
- **CHUNK_SIZE = 128**로 시작합니다(GRPO MMMU test가 256에서 엔진이 죽었던 일 반영 — 호스트 메모리 여유).

> **실행:** 런타임 **A100 GPU** → **런타임 > 모두 실행**. 40GB가 아니면 첫 코드 셀에서 멈춥니다. **3번 셀이 도는 동안 다른 셀을 실행하지 마세요.** 마지막 셀이 런타임을 자동 종료합니다.
>
> **Drive에 새로 올릴 것: 없습니다.** GRPO 전체 평가 때 올린 `src/mmmu_final/`(6개)와 `configs/final_eval_ids_full.json`을 그대로 씁니다(첫 코드 셀이 해시 확인).

In [1]:
from google.colab import drive
from pathlib import Path
import os, sys, json, re, subprocess, shutil, hashlib, time

drive.mount('/content/drive')

# === 사용자 설정: 이 값만 바꿉니다 ===
PART = 'pro'   # 'mmmu_test' (공용 계정: MMMU test 10,500) 또는 'pro' (팀원 계정: MMMU-Pro standard + vision 1,730씩)
CHUNK_SIZE = 128   # 묶음 크기(저장 단위). 바꾸지 않아도 됩니다.

assert PART in ('mmmu_test', 'pro'), "첫 줄의 PART를 'mmmu_test'(공용 계정) 또는 'pro'(팀원 계정)로 정하세요."
SETS = {'mmmu_test': ['mmmu_test'], 'pro': ['pro_standard', 'pro_vision']}[PART]
MODEL_TAG = 'baseline'
COMPARE_TAG = 'grpo_v4_step170'   # 같은 폴더의 GRPO 결과와 비교
N_EXPECTED = {'mmmu_test': 10500, 'pro_standard': 1730, 'pro_vision': 1730}

PROJECT = Path('/content/drive/MyDrive/MMDL/code/qwen3_vl_mmmu_eval')
CONFIG = PROJECT/'configs'/'final_16k.yaml'
IDS = PROJECT/'configs'/'final_eval_ids_full.json'
REQ = PROJECT/'requirements.txt'
BASELINE_RUN_DIR = PROJECT/'outputs'/'final_seed3407_16k'                            # chat template 비교용 (val baseline)
MMMU_CACHE = Path('/content/drive/MyDrive/MMDL/pmc_vqa/eval/mmmu_dataset_cache')   # MMMU test는 기존 Drive 캐시
PRO_CACHE = Path('/content/mmmu_pro_cache')                                     # MMMU-Pro는 Colab 로컬
OUTPUT_ROOT = Path('/content/drive/MyDrive/MMDL/pmc_vqa/eval/mmmu_eval_outputs/final_eval_full_a100_40gb')
EVAL_VENV = Path('/content/mmmu_eval_venv')
REQUIRED_GPU = 'NVIDIA A100-SXM4-40GB'
BASE_MODEL_ID, BASE_REVISION = 'Qwen/Qwen3-VL-4B-Instruct', 'ebb281ec70b05090aa6165b016eac8ec08e71b17'

# --- 고정 파일 확인: GRPO 전체 평가와 같은 설정·문항 ID·mmmu_final 코드인지 ---
EXPECTED_SHA = {'final_16k.yaml': 'a6b0ac30c57e5d2a4358233060cd45ff642a3ba2193eeeecbeea45486334d818',
                'final_eval_ids_full.json': 'b0fdacad4993c1556eee93395a202be86fcf9ea4e3a6f25011288a53360d4c66'}
for p in (CONFIG, IDS, REQ, PROJECT/'src'/'mmmu_final'/'run.py', BASELINE_RUN_DIR/'chat_template.jinja'):
    assert p.is_file(), f'필요한 파일이 없습니다: {p}  → Drive의 평가 프로젝트를 확인하세요(팀원 계정은 공유 폴더 바로가기)'
for p in (CONFIG, IDS):
    h = hashlib.sha256(p.read_bytes()).hexdigest()
    assert h == EXPECTED_SHA[p.name], f'{p.name}이 고정본과 다릅니다: {h}'
EXPECTED_FINAL_SRC = {'__init__.py': '39a51674e8b071b4def2f6385ac6d4fa6db778d4b8266558aae4f82688e1a382', 'compare.py': 'c7dd273e39a202969d946cb788dae4500714fd9cea1a112ef509f8ef86166b19', 'official_pro.py': 'd3f47f4440f46d132f77b2f708593fccd9aa9ded6892f9e0677e3a9c0df00cfb', 'run.py': '90c6aa73eccfd806f91028d2447be139b667aac8467c58fecae983675746a3bd', 'scoring.py': '8765a250185fa97234804abb58d2900a2261e0934d168a20c9cd59c1a75531f1', 'sets.py': '064d1061b23ee1d1005d3d1b8094b0797b8ee697d2efcbc85be74566d5b08e0d'}
got = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in (PROJECT/'src'/'mmmu_final').glob('*.py')}
bad = sorted(n for n in EXPECTED_FINAL_SRC if got.get(n) != EXPECTED_FINAL_SRC[n])
assert not bad, f'src/mmmu_final/의 다음 파일이 GRPO 전체 평가 때와 다릅니다(또는 없음): {bad}'
print('✔ 평가 설정·전체 문항 ID·mmmu_final 코드 = GRPO 전체 평가와 같은 고정본')

import torch
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
assert gpu_name == REQUIRED_GPU, (f'최종 평가는 {REQUIRED_GPU}에서만 돌립니다. 현재: {gpu_name}\n'
    '→ 런타임 > 런타임 연결 해제 및 삭제 후 다시 연결해 40GB를 받으세요.')

# --- 결과 폴더 쓰기 확인 (팀원 계정은 공유 폴더에 편집 권한이 있어야 함) ---
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
probe = OUTPUT_ROOT/f'.write_test_baseline_{PART}'; probe.write_text('ok'); probe.unlink()
# --- 이전 시도가 시작 직후 끊겨 잠금 파일만 남은 세트 폴더 정리 (평가 프로세스가 없을 때만) ---
for s in SETS:
    d = OUTPUT_ROOT/MODEL_TAG/s
    if d.is_dir() and {p.name for p in d.iterdir()} <= {'.run.lock'}:
        running = subprocess.run(['pgrep', '-f', 'mmmu_final[.]run'], capture_output=True).returncode == 0
        assert not running, '평가 프로세스가 아직 돌고 있습니다. !pkill -f mmmu_final.run 으로 종료한 뒤 다시 실행하세요.'
        shutil.rmtree(d); print('잠금 파일만 남은 폴더를 지웠습니다:', d)
    elif (d/'run_meta.json').is_file():
        m = json.loads((d/'run_meta.json').read_text())
        n = sum(1 for _ in open(d/'raw_outputs.jsonl', encoding='utf-8')) if (d/'raw_outputs.jsonl').is_file() else 0
        print(f'[{s}] 이전 실행 기록 있음: 저장된 응답 약 {n}/{N_EXPECTED[s]}, 상태 {m.get("status")} → 이어서 진행')
    g = OUTPUT_ROOT/COMPARE_TAG/s/'scores.json'
    print(f'[{s}] GRPO 결과:', '있음' if g.is_file() else '아직 없음(다른 계정·세션에서 진행 중이면 정상)')
print('GPU:', gpu_name, '| PART:', PART, SETS, '| 결과:', OUTPUT_ROOT/MODEL_TAG)

Mounted at /content/drive
✔ 평가 설정·전체 문항 ID·mmmu_final 코드 = GRPO 전체 평가와 같은 고정본
[pro_standard] 이전 실행 기록 있음: 저장된 응답 약 338/1730, 상태 generating → 이어서 진행
[pro_standard] GRPO 결과: 있음
[pro_vision] GRPO 결과: 있음
GPU: NVIDIA A100-SXM4-40GB | PART: pro ['pro_standard', 'pro_vision'] | 결과: /content/drive/MyDrive/MMDL/pmc_vqa/eval/mmmu_eval_outputs/final_eval_full_a100_40gb/baseline


## 1. 평가 전용 가상환경 (기존 평가 노트북과 같은 셀)

In [2]:
def checked(command, *, cwd=None, env=None):
    print('$', ' '.join(map(str, command)), flush=True)
    return subprocess.run([str(v) for v in command], cwd=cwd, env=env, check=True)

EVAL_PY = EVAL_VENV/'bin'/'python'
marker = EVAL_VENV/'.installed_ok'
req_hash = hashlib.sha256(REQ.read_bytes()).hexdigest()
if not (marker.is_file() and marker.read_text() == req_hash):
    shutil.rmtree(EVAL_VENV, ignore_errors=True)
    checked([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
    checked([sys.executable, '-m', 'uv', 'venv', '--python', '3.12', str(EVAL_VENV)])
    print('requirements:\n' + REQ.read_text(encoding='utf-8'))
    checked([sys.executable, '-m', 'uv', 'pip', 'install', '--python', str(EVAL_PY), '-r', str(REQ)])
    marker.write_text(req_hash)
out = subprocess.run([str(EVAL_PY), '-c',
    'import sys,torch,vllm,transformers,datasets,qwen_vl_utils,importlib.metadata as m;'
    'print(sys.version.split()[0], torch.__version__, vllm.__version__, transformers.__version__, '
    'datasets.__version__, m.version("qwen-vl-utils"), torch.cuda.is_available())'],
    capture_output=True, text=True, check=True).stdout.strip()
print('평가 환경 (python torch vllm transformers datasets qwen-vl-utils cuda):', out)
pyv, torchv, vllmv, tfv, dsv, qvlv, cuda = out.split()
assert (torchv.split('+')[0], vllmv, tfv, dsv, qvlv) == ('2.8.0', '0.11.0', '4.57.1', '4.2.0', '0.0.14'), 'baseline 실행 환경과 패키지 버전이 다릅니다'
assert cuda == 'True', '평가 환경에서 CUDA를 찾지 못했습니다'
print('✔ baseline과 같은 패키지 버전')

$ /usr/bin/python3 -m pip install -q uv
$ /usr/bin/python3 -m uv venv --python 3.12 /content/mmmu_eval_venv
requirements:
# Candidate pins; NOT yet GPU-validated. See README and Colab verification.
# After successful A100 smoke/resume tests, capture the installed environment
# using scripts/freeze_verified_environment.py and review/commit the lock.
vllm==0.11.0
torch==2.8.0
torchvision==0.23.0
torchaudio==2.8.0
transformers==4.57.1
qwen-vl-utils==0.0.14
datasets==4.2.0
Pillow==11.3.0
PyYAML==6.0.3
pytest==8.4.2

$ /usr/bin/python3 -m uv pip install --python /content/mmmu_eval_venv/bin/python -r /content/drive/MyDrive/MMDL/code/qwen3_vl_mmmu_eval/requirements.txt
평가 환경 (python torch vllm transformers datasets qwen-vl-utils cuda): 3.12.3 2.8.0+cu128 0.11.0 4.57.1 4.2.0 0.0.14 True
✔ baseline과 같은 패키지 버전


## 2. base 모델 확인 (병합 없음)
고정 revision의 processor를 받아 chat template이 val baseline(63.89%) 실행 때와 같은지 확인합니다. 가중치는 3번 셀에서 vLLM이 받습니다.

In [3]:
tmpl = subprocess.run([str(EVAL_PY), '-c',
    'import sys;from transformers import AutoProcessor;'
    'sys.stdout.write(AutoProcessor.from_pretrained(sys.argv[1], revision=sys.argv[2]).chat_template)', BASE_MODEL_ID, BASE_REVISION],
    capture_output=True, text=True)
assert tmpl.returncode == 0, tmpl.stderr[-2000:]
assert tmpl.stdout == (BASELINE_RUN_DIR/'chat_template.jinja').read_text(encoding='utf-8'), 'base 모델의 chat template이 val baseline 실행 때와 다릅니다.'
print('✔ 모델:', BASE_MODEL_ID, '@', BASE_REVISION[:12], '| chat template = val baseline (sha256', hashlib.sha256(tmpl.stdout.encode()).hexdigest()[:16] + ')')

✔ 모델: Qwen/Qwen3-VL-4B-Instruct @ ebb281ec70b0 | chat template = val baseline (sha256 3636d0f0bd6bef02)


## 3. 세트별 생성 + 채점
`PART`의 세트를 차례로 `python -m mmmu_final.run`으로 실행합니다(묶음마다 저장, 끊기면 다시 실행해서 이어 감). 전체 로그는 `OUTPUT_ROOT/baseline/<세트>/stdout.log`.

In [4]:
env = os.environ.copy()
env['PYTHONPATH'] = str(PROJECT/'src'); env['PYTHONUNBUFFERED'] = '1'; env['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
progress = re.compile(r'\d+%\|')
for s in SETS:
    out_dir = OUTPUT_ROOT/MODEL_TAG/s
    if (out_dir/'scores.json').is_file():
        sc = json.loads((out_dir/'scores.json').read_text())
        print(f'[{s}] 이미 채점 완료 → 건너뜀: {sc["correct"]}/{sc["n"]}'); continue
    out_dir.mkdir(parents=True, exist_ok=True)
    cmd = [str(EVAL_PY), '-m', 'mmmu_final.run', '--set', s, '--model_tag', MODEL_TAG,
           '--model_path', BASE_MODEL_ID, '--model_revision', BASE_REVISION,
           '--config', str(CONFIG), '--ids', str(IDS), '--output_root', str(OUTPUT_ROOT),
           '--data_root', str(MMMU_CACHE if s == 'mmmu_test' else PRO_CACHE), '--seed', '3407',
           '--require_gpu', 'A100-SXM4-40GB', '--chunk_size', str(CHUNK_SIZE)]
    print(f'\n=== [{s}] 시작 ({N_EXPECTED[s]}문항) ===', flush=True); t0 = time.time(); last = 0.0
    with open(out_dir/'stdout.log', 'a', encoding='utf-8') as log:
        proc = subprocess.Popen(cmd, cwd=PROJECT, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            log.write(line); log.flush()
            if progress.search(line) or line.startswith('Preflight') and 'questions' not in line:
                if time.time() - last < 120: continue
                last = time.time()
            print(line, end='', flush=True)
        rc = proc.wait()
    print(f'[{s}] 이번 세션 {(time.time()-t0)/3600:.2f}시간, 종료 코드 {rc}')
    if rc != 0:
        raise RuntimeError(f'[{s}] 평가 실패 (exit {rc}). 로그: {out_dir/"stdout.log"} — 끊긴 것이면 같은 PART로 처음부터 다시 실행하면 이어 갑니다.')
print('\n✔ 완료:', SETS)


=== [pro_standard] 시작 (1730문항) ===
GPU: NVIDIA A100-SXM4-40GB

Generating test split:   0%|          | 0/1730 [00:00<?, ? examples/s]
Preflight: reusing cached input_lengths.json (1730 questions)
Preflight: 1730 questions, longest input 9487 tokens -> max_model_len 26624
INFO 11-22 05:02:55 [__init__.py:216] Automatically detected platform cuda.
INFO 11-22 05:02:58 [utils.py:233] non-default args: {'dtype': 'bfloat16', 'seed': 3407, 'max_model_len': 26624, 'disable_log_stats': True, 'revision': 'ebb281ec70b05090aa6165b016eac8ec08e71b17', 'limit_mm_per_prompt': {'image': 10}, 'generation_config': 'vllm', 'model': 'Qwen/Qwen3-VL-4B-Instruct'}
INFO 11-22 05:03:11 [model.py:547] Resolved architecture: Qwen3VLForConditionalGeneration
`torch_dtype` is deprecated! Use `dtype` instead!
INFO 11-22 05:03:11 [model.py:1510] Using max model len 26624
INFO 11-22 05:03:11 [scheduler.py:205] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 11-22 05:03:18 [__init__.py:216] Automatica

## 4. 결과 요약 · GRPO와 같은 문항 비교
baseline·GRPO 결과가 둘 다 있는 세트만 비교합니다(`final_comparison_full.md/json`, 표의 'submission' 열 = GRPO step 170). GPU·seed·문항 ID가 다르면 멈춥니다.

In [5]:
for tag in (MODEL_TAG, COMPARE_TAG):
    for s in N_EXPECTED:
        f = OUTPUT_ROOT/tag/s/'scores.json'
        if not f.is_file():
            print(f'{tag:<16}{s:<13} (결과 없음 — 다른 계정에서 진행 중이거나 미실행)'); continue
        sc = json.loads(f.read_text())
        assert sc['n'] == N_EXPECTED[s], sc['n']
        extra = (f" | 공식(무작위 포함, 참고) {sc['reference_official_with_guess']['correct']} | 출처별 {sc['by_origin']}" if 'by_origin' in sc else
                 f" | 객관식/주관식 {sc['question_type']}")
        print(f"{tag:<16}{s:<13} {sc['correct']}/{sc['n']} = {100*sc['accuracy']:.2f}% | 잘림 {sc['trunc_pct']:.1f}% | "
              f"중앙 {sc['median_output_tokens']:.0f}토큰 | max_model_len {sc['max_model_len_used']} | {sc['gpu']}{extra}")
both = [s for s in N_EXPECTED if all((OUTPUT_ROOT/t/s/'scores.json').is_file() for t in (MODEL_TAG, COMPARE_TAG))]
if both:
    print('\n비교할 세트:', both)
    cmp_code = ('import sys, shutil; from pathlib import Path; from mmmu_final.compare import compare; '
                'r = Path(sys.argv[1]); compare(r, "baseline", "grpo_v4_step170"); '
                '[shutil.move(r/f"final_comparison.{x}", r/f"final_comparison_full.{x}") for x in ("json", "md")]')
    r = subprocess.run([str(EVAL_PY), '-c', cmp_code, str(OUTPUT_ROOT)], cwd=PROJECT, env=env, capture_output=True, text=True)
    print(r.stdout); assert r.returncode == 0, r.stderr[-2000:]
    print('저장:', OUTPUT_ROOT/'final_comparison_full.md', '(submission 열 = GRPO step 170)')
else:
    print('\n같은 세트의 baseline·GRPO 결과가 아직 함께 없습니다. 나중에 이 셀만 다시 실행하면 비교표가 만들어집니다.')

baseline        mmmu_test     (결과 없음 — 다른 계정에서 진행 중이거나 미실행)
baseline        pro_standard  888/1730 = 51.33% | 잘림 14.2% | 중앙 739토큰 | max_model_len 26624 | NVIDIA A100-SXM4-40GB | 공식(무작위 포함, 참고) 923 | 출처별 {'test': {'n': 1153, 'correct': 609}, 'validation': {'n': 577, 'correct': 279}}
baseline        pro_vision    841/1730 = 48.61% | 잘림 13.8% | 중앙 888토큰 | max_model_len 24576 | NVIDIA A100-SXM4-40GB | 공식(무작위 포함, 참고) 884 | 출처별 {'test': {'n': 1153, 'correct': 566}, 'validation': {'n': 577, 'correct': 275}}
grpo_v4_step170 mmmu_test     (결과 없음 — 다른 계정에서 진행 중이거나 미실행)
grpo_v4_step170 pro_standard  937/1730 = 54.16% | 잘림 3.3% | 중앙 716토큰 | max_model_len 26624 | NVIDIA A100-SXM4-40GB | 공식(무작위 포함, 참고) 945 | 출처별 {'test': {'n': 1153, 'correct': 634}, 'validation': {'n': 577, 'correct': 303}}
grpo_v4_step170 pro_vision    881/1730 = 50.92% | 잘림 3.8% | 중앙 890토큰 | max_model_len 24576 | NVIDIA A100-SXM4-40GB | 공식(무작위 포함, 참고) 894 | 출처별 {'test': {'n': 1153, 'correct': 596}, 'validation': {'n': 577, 'correc

## 5. 결과 보존
Drive의 `mmmu_eval_outputs/final_eval_full_a100_40gb/` 폴더 전체를 로컬 `MMDL/pmc_vqa/eval/mmmu_eval_outputs/`로 가져와 주세요. 응답 원문(`raw_outputs.jsonl`)에는 MMMU·MMMU-Pro 문항 내용이 들어 있으니 공개 저장소에는 집계 파일(`scores.json`, `final_comparison_full.*`)만 올립니다.

In [6]:
import time
from google.colab import runtime
print('모든 셀 완료. 60초 뒤 런타임을 종료합니다. 중단하려면 셀을 정지하세요.')
time.sleep(60)
runtime.unassign()

모든 셀 완료. 60초 뒤 런타임을 종료합니다. 중단하려면 셀을 정지하세요.
